# LLaVA Evaluation Notebook (Colab Ready)


This notebook reproduces the `test_llava.py` pipeline end-to-end:

- Load dataset from `data.json`

- Run visual-only and context-aware inference

- Reallocate attention using CLIP similarity

- Export results to `model_answers.csv`



## Files you should have in Colab runtime

- `data.json`

- `images/` folder with referenced images



If models are gated/private, add a Hugging Face token as a Colab secret named `HF_TOKEN`.


In [ ]:
# Colab dependency setup
!pip -q install -U transformers pillow accelerate sentence-transformers pandas python-dotenv huggingface_hub

In [ ]:
# Optional: authenticate to Hugging Face using Colab secret HF_TOKEN
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(hf_token)
        print('HF login successful.')
    else:
        print('No HF_TOKEN secret found. Continuing without login.')
except Exception as exc:
    print(f'Colab secret access not available: {exc}')
    print('Continuing without login.')

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from sentence_transformers import SentenceTransformer, util
from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration

In [ ]:
# Core model + response helpers (from load_llava.py and inference.py)
def load_model(model_id: str = "llava-hf/llava-onevision-qwen2-0.5b-ov-hf"):
    model = LlavaOnevisionForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
    ).to("cpu")

    processor = AutoProcessor.from_pretrained(model_id)
    return model, processor


def generate_response(model, processor, image_path, question, context=None):
    if context:
        prompt = f"{context}. So {question}"
    else:
        prompt = question

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    rendered_prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    raw_image = Image.open(image_path).convert("RGB")
    inputs = processor(images=raw_image, text=rendered_prompt, return_tensors="pt").to("cpu", torch.float32)

    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        return_dict_in_generate=True,
        output_scores=True,
    )

    prompt_length = inputs["input_ids"].shape[1]
    generated_tokens = output.sequences[:, prompt_length:]
    response = processor.decode(generated_tokens[0], skip_special_tokens=True)

    transition_scores = model.compute_transition_scores(
        output.sequences,
        output.scores,
        normalize_logits=True,
    )
    avg_score = transition_scores.mean().item()

    return response, avg_score, output.scores

In [ ]:
# test_llava.py pipeline helpers
DEFAULT_OUTPUT = "model_answers.csv"


def load_examples(data_path):
    with open(data_path, "r", encoding="utf-8") as file:
        return json.load(file)


def calculate_similarity(clip_model, question, context, image_path, visual_priority):
    q_emb = clip_model.encode(question)
    context_emb = clip_model.encode(context)

    image = Image.open(image_path).convert("RGB")
    table_emb = clip_model.encode(image)

    context_score = util.cos_sim(q_emb, context_emb).item()
    table_score = util.cos_sim(q_emb, table_emb).item()

    context_weight = max(context_score, 0.0)
    table_weight = max(table_score, 0.0) * visual_priority
    total_weight = context_weight + table_weight

    if total_weight == 0:
        return 0.5, 0.5

    lambda_context = context_weight / total_weight
    lambda_table = table_weight / total_weight
    return lambda_context, lambda_table


def choose_final_answer(visual_answer, context_answer, lambda_table, lambda_context):
    if lambda_table >= lambda_context:
        return visual_answer
    return context_answer


def generate_fusion_attn_answer(
    model,
    processor,
    clip_model,
    image_path,
    question,
    context,
    visual_priority,
    visual_answer=None,
 ):
    if visual_answer is None:
        visual_answer, _, _ = generate_response(model, processor, image_path, question)

    context_answer, _, _ = generate_response(
        model,
        processor,
        image_path,
        question,
        context,
    )

    lambda_context, lambda_table = calculate_similarity(
        clip_model,
        question,
        context,
        image_path,
        visual_priority,
    )

    return choose_final_answer(
        visual_answer,
        context_answer,
        lambda_table,
        lambda_context,
    )


def attn_realloc_ans(
    image_path,
    question,
    context,
    visual_priority=3.0,
    model=None,
    processor=None,
    similarity_model=None,
):
    if model is None or processor is None:
        model, processor = load_model()

    if similarity_model is None:
        similarity_model = SentenceTransformer("clip-ViT-B-32")

    return generate_fusion_attn_answer(
        model,
        processor,
        similarity_model,
        image_path,
        question,
        context,
        visual_priority,
    )

In [ ]:
# Runtime configuration
DATA_PATH = Path("data.json")
IMAGES_DIR = Path("images")
OUTPUT_PATH = Path(DEFAULT_OUTPUT)
VISUAL_PRIORITY = 3.0
LIMIT = None  # Set to an int like 5 for a quick smoke test

print(f"Data path exists: {DATA_PATH.exists()}")
print(f"Images dir exists: {IMAGES_DIR.exists()}")

In [ ]:
# Run evaluation + save CSV
examples = load_examples(DATA_PATH)
selected_examples = examples if LIMIT is None else examples[:LIMIT]

model, processor = load_model()
similarity_model = SentenceTransformer("clip-ViT-B-32")
rows = []

for idx, example in enumerate(selected_examples, start=1):
    image_path = IMAGES_DIR / example["image"]
    question = example["question"]
    context = example["context"]

    print(f"[{idx}/{len(selected_examples)}] id={example.get('id')} image={example['image']}")

    without_attn_realloc, _, _ = generate_response(
        model,
        processor,
        image_path,
        question,
    )

    with_attn_realloc = attn_realloc_ans(
        image_path=image_path,
        question=question,
        context=context,
        visual_priority=VISUAL_PRIORITY,
        model=model,
        processor=processor,
        similarity_model=similarity_model,
    )

    rows.append(
        {
            "image_path": str(image_path),
            "question": question,
            "context": context,
            "actual_answer": example.get("answer"),
            "without_attn_realloc": without_attn_realloc,
            "with_attn_realloc": with_attn_realloc,
        }
    )

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"Saved {len(df)} rows to {OUTPUT_PATH}")
df.head()

## Notes

- For local runs, keep script device as CPU (`--device cpu`).

- For Colab T4 runs, use the runner cells with `--device cuda`.

- Start with a small `--limit` for smoke tests, then remove limit for full evaluation.

## Colab T4 Script Runner

Use this section when running the project scripts directly on a Colab T4 runtime.

Expected files in runtime: `test_llava.py`, `test_qwen.py`, `fusion.py`, `pipeline.py`, `inference.py`, `load_llava.py`, `load_qwen.py`, `realloc_attn.py`, `data.json`, and `images/`.

In [ ]:
# Verify GPU runtime (T4 expected)
!nvidia-smi

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Run LLaVA experiment script with corrected mitigation on Colab GPU
!python -m test_llava --fusion-mode phase2 --visual-priority 3.0 --device cuda --limit 5 --output model_answers_colab.csv

In [ ]:
# Run Qwen experiment script with corrected mitigation on Colab GPU
!python -m test_qwen --fusion-mode phase2 --visual-priority 3.0 --device cuda --limit 5 --output qwen_answers_colab.csv

In [ ]:
# Preview generated files
import pandas as pd

llava_df = pd.read_csv('model_answers_colab.csv')
qwen_df = pd.read_csv('qwen_answers_colab.csv')

print('LLaVA rows:', len(llava_df))
print('Qwen rows:', len(qwen_df))
display(llava_df.head(3))
display(qwen_df.head(3))